# Exercice 1 - Probabilité implicite de défaut

On cherche la proba de défaut annuelle constante implicite dans le prix de l'obligation.

In [ ]:
import numpy as np
from scipy.optimize import brentq

In [ ]:
nominal = 100
coupon = 0.04
c = coupon / 2 * nominal  # coupon semestriel = 2
y = 0.05  # rendement continu
T = 4
r = 0.03  # taux sans risque composé annuel
R = 0.30  # recovery

In [ ]:
# dates de coupon semestrielles
dates = np.arange(0.5, T + 0.5, 0.5)

# prix de marché en actualisant au rendement continu
prix_marche = sum(c * np.exp(-y*t) for t in dates) + nominal * np.exp(-y*T)
print(f"Prix de marché = {prix_marche:.4f}")

In [ ]:
# facteur d'actualisation sans risque
def DF(t):
    return (1 + r)**(-t)

# prix sans risque de defaut
prix_rf = sum(c * DF(t) for t in dates) + nominal * DF(T)
print(f"Prix risk-free = {prix_rf:.4f}")
print(f"Ecart = {prix_rf - prix_marche:.2f}")

## Modélisation

Le défaut arrive en fin d'année (j = 1,2,3,4). Si pas de defaut avant l'année j, la proba de faire défaut cette année là c'est p. Si defaut à la date j on recupère R * nominal, sinon on touche les coupons normalement.

In [ ]:
def prix_avec_defaut(p):
    prix = 0
    
    # coupons : on les reçoit tant qu'il n'y a pas eu de défaut
    for t in dates:
        annee = int(np.ceil(t))
        if t == int(t):  # fin d'année : reçu ssi survie cette année
            surv = (1-p)**int(t)
        else:  # mi-année : reçu ssi survie année précédente
            surv = (1-p)**(annee - 1)
        prix += c * surv * DF(t)
    
    # principal si pas de defaut du tout
    prix += nominal * (1-p)**T * DF(T)
    
    # recouvrement si defaut en année j
    for j in range(1, T+1):
        p_def_j = (1-p)**(j-1) * p
        prix += R * nominal * p_def_j * DF(j)
    
    return prix

In [ ]:
# on résout pour trouver p
p_defaut = brentq(lambda p: prix_avec_defaut(p) - prix_marche, 1e-4, 0.5)
print(f"Probabilité de défaut annuelle : p = {p_defaut*100:.2f}%")

In [ ]:
# verif rapide avec la formule approchée : p ≈ spread / (1-R)
rc = np.log(1 + r)  # taux sans risque en continu
p_approx = (y - rc) / (1 - R)
print(f"Approximation : p ≈ {p_approx*100:.2f}%")
print(f"Valeur exacte  : p = {p_defaut*100:.2f}%")
print("Les deux sont cohérentes.")

## Interprétation

On obtient environ 2.9% de proba de défaut par an. C'est cohérent : le spread (5% - 3% ≈ 2%) divisé par (1-R) = 0.7 donne à peu pres ce résultat.

C'est une proba risk-neutral (pas historique), donc elle inclut une prime de risque et surestime la "vraie" probabilité de défaut.